In [3]:
import gymnasium as gym
import minigrid

from IPython.display import Video, display, HTML
import os

def show_output(outputs, video_path):
    from IPython.display import display, HTML
    
    text_output = "<br>".join(outputs)
    display(HTML(f"""
    <div style="display: flex; gap: 20px;">
        <div style="flex: 1; max-height: 600px; overflow-y: scroll; font-family: monospace; white-space: pre;">
{text_output}
        </div>
        <div style="flex: 1;">
            <video controls width="100%">
                <source src="{video_path}" type="video/mp4">
            </video>
        </div>
    </div>
    """))

def show_output_with_latest(outputs):
    video_files = [f for f in os.listdir("videos/") if f.endswith('.mp4')]
    if video_files:
        latest_video = os.path.join("videos/", sorted(video_files)[-1])
        show_output(outputs, latest_video)

In [6]:
import torch
import torch.nn as nn
import numpy as np

class Policy(nn.Module):
    def __init__(self, obs_size, action_size, hidden_size=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size)
        )
    
    def forward(self, x):
        return self.net(x)
    
    def get_action(self, obs):
        logits = self.forward(obs)
        probs = torch.softmax(logits, dim=-1)
        action_tensor = torch.multinomial(probs, 1)
        action = int(action_tensor.item())
        log_prob = torch.log_softmax(logits, dim=-1)[action_tensor.squeeze()]
        return action, log_prob

In [ ]:
# create env
env = gym.make("BabyAI-OpenRedBlueDoorsDebug-v0", render_mode="rgb_array")
env = gym.wrappers.RecordVideo(env, "videos/", episode_trigger=lambda x: True)

# initialize policy
policy = Policy(obs_size=7*7*3+4, action_size=env.action_space.n)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

def process_obs(obs):
    img_flat = obs['image'].flatten().astype(np.float32)
    dir_onehot = np.zeros(4, dtype=np.float32)
    dir_onehot[obs['direction']] = 1
    return torch.FloatTensor(np.concatenate([img_flat, dir_onehot]))

# reset and collect one episode
observation, info = env.reset()
states, actions, rewards, log_probs = [], [], [], []
outputs = []

for step in range(100):
    # render observation
    lines = []
    lines.append(f"Step {step}")
    lines.append(f"Mission: {observation['mission']}")
    lines.append(f"Direction: {observation['direction']}")
    lines.append("")
    
    img = observation['image']
    for row in img:
        lines.append(" ".join(f"{pixel[0]:2d}" for pixel in row))
    
    outputs.append("\n".join(lines))
    
    # get action from policy
    state = process_obs(observation)
    action, log_prob = policy.get_action(state)
    
    # store trajectory
    states.append(state)
    actions.append(action)
    log_probs.append(log_prob)
    
    observation, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    
    if terminated or truncated:
        break

env.close()
show_output_with_latest(outputs)

print(f"Episode length: {len(rewards)}")
print(f"Total reward: {sum(rewards)}")

ResetNeeded: Cannot call env.step() before calling env.reset()